In [49]:
import os
import time

from dotenv import load_dotenv
from langchain_core.prompts import PromptTemplate
from langchain_community.embeddings import HuggingFaceEmbeddings
from langchain_community.document_loaders import PyPDFLoader, DirectoryLoader  
from langchain_text_splitters import RecursiveCharacterTextSplitter  
# from langchain_pinecone import PineconeVectorStore
# from pinecone import Pinecone, ServerlessSpec
from langchain_community.vectorstores import Pinecone
from langchain_community.llms import CTransformers  # ← changed

In [50]:

# from langchain_community.chains import RetrievalQA

In [51]:
#Extract data from the PDF
def load_pdf(data):
    loader = DirectoryLoader(data,
                    glob="*.pdf",
                    loader_cls=PyPDFLoader)
    
    documents = loader.load()

    return documents

In [39]:
extracted_data = load_pdf("../data/")

In [40]:
# extracted_data

In [ ]:
# Create text chunks

def text_split(extracted_data):
    text_splitter = RecursiveCharacterTextSplitter(chunk_size = 500, chunk_overlap = 20)
    text_chuks = text_splitter.split_documents(extracted_data)
    
    return text_chuks
    

In [42]:
text_chunks = text_split(extracted_data)
print(f'length of text chunks: {len(text_chunks)}')

length of text chunks: 5860


In [43]:
# Create embeddings and store in Pinecone
def download_embedding_model():
    embeddings = HuggingFaceEmbeddings(model_name="sentence-transformers/all-MiniLM-L6-v2")
    return embeddings

In [44]:
embeddings = download_embedding_model()

Loading weights: 100%|██████████| 103/103 [00:00<00:00, 5576.02it/s]
BertModel LOAD REPORT from: sentence-transformers/all-MiniLM-L6-v2
Key                     | Status     |  | 
------------------------+------------+--+-
embeddings.position_ids | UNEXPECTED |  | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.


In [61]:
from pinecone import Pinecone, ServerlessSpec
import os

api_key = os.getenv("PINECONE_API_KEY")

# Initialize Pinecone
pc = Pinecone(api_key=api_key)

In [67]:
from langchain_pinecone import PineconeVectorStore
docsearch = PineconeVectorStore.from_documents(
    documents=text_chunks,  # ← Document objects, not raw strings
    embedding=embeddings,
    index_name="medical-chatbot"
)

In [68]:
docsearch